# Player Tracking in Sports - Multi-View Tracking and 3D Reconstruction

## Imports

In [ ]:
from src.utils.video import open_video, get_frames, save_video, produce_detection_output_video
from src.tracking.motion_detection import MOG2_motion_detection, refine_blobs
from src.tracking.image_processing import opening_closing, normalize_illumination
from src.utils.visualization import show_image, show_images
from src.detection.yolo_detection import run_yolo_detection, yolo_to_detection_output

import time
import cv2
import logging
from ultralytics import YOLO

# Suppress YOLO logging
logging.getLogger("ultralytics").setLevel(logging.WARNING)
logging.getLogger("ultralytics.hub.utils").setLevel(logging.WARNING)

In [ ]:
CURRENT_CAMERA_ID = "cam_13"

## Path Definitions

In [ ]:
VIDEOS_DIR = "data/videos"
CAMERA_SETTINGS_DIR = "data/camera_settings"

# Define fundamental paths for each camera
CAMERAS = {
    "cam_13": {
        "video_path": f"{VIDEOS_DIR}/out13.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam13_settings.json",
    },
    "cam_2": {
        "video_path": f"{VIDEOS_DIR}/out2.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam2_settings.json",
    },
    "cam_4": {
        "video_path": f"{VIDEOS_DIR}/out4.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam4_settings.json",
    },
}

## Open Video and Read Frames

In [ ]:
# Currently just one camera
cap = open_video(CAMERAS[CURRENT_CAMERA_ID]["video_path"])
frames_color, _ = get_frames(cap, max_frames=None)

# Release the video capture object to free resources
cap.release()

show_images(frames_color)

## Image Preprocessing

We first apply a Gaussian blur to reduce reflection and noise in the frames. This helps improve the performance of motion detection algorithms.

Then, we apply a custom function `remove_reflections` to further clean the frames by removing reflections and shadows that can interfere with motion detection.

In [ ]:
# Function to remove reflections and shadows from the motion masks
refined_frames_color = normalize_illumination(frames_color, clip_limit=6.0, tile_grid_size=(64, 32))

show_image(refined_frames_color[0], title="Preprocessed Frame")

## Motion Detection

### Mixture of Gaussians (MOG2) Background Subtraction

In [ ]:
# Parameters for MOG2 motion detection
learning_rate = -1 # Use default learning rate (auto-adaptive)
history_length = 1000
var_threshold = 15
detect_shadows = False

# Compute motion masks using MOG2
masks = MOG2_motion_detection(refined_frames_color, learning_rate, history_length, var_threshold, detect_shadows)

### Motion Video Post-processing

- Apply morphological operations to clean up the masks.
- Blob filtering to remove small noise blobs and keep only significant motion areas.

In [ ]:
# Morphological opening and closing to clean up the masks
opening_kernel_size = 7
closing_kernel_size = 7
masks = opening_closing(masks, opening_kernel_size=opening_kernel_size, closing_kernel_size=closing_kernel_size)

# Blob filtering to remove small noise blobs and keep only significant motion areas
# masks = refine_blobs(masks, min_area=200)

In [ ]:
# Save results as video
save_video(masks, "results/motion_detection/cam13/cam_13_mog2_masks.mp4")

**Temporary** : Cam 13 — Region of Interest

In [ ]:
# Adjust roi_y to find the right horizontal boundary for cam 13
roi_y = 300
frame = frames_color[0]  # Use the first frame for visualization
h, w, _ = frame.shape

cv2.line(frame, (0, roi_y), (w, roi_y), (255, 255, 255), 2)

## Using YOLOv8 for Player Detection
We could also use a deep learning-based object detection model like YOLOv8 to detect players in each frame. This can be done by loading a pre-trained YOLOv8 model and running inference on the frames to get bounding boxes for detected players.

In [ ]:
model = YOLO("./models/yolov8m.pt") # Load the pre-trained YOLOv8m model

### Run YOLO Detection on All Frames

Use YOLOv8's built-in `.predict()` method to detect objects in each frame. The output will include bounding boxes, confidence scores, and class labels for each detected object.

**Output per frame:**
- Bounding boxes: `[x1, y1, x2, y2]` (pixel coordinates)
- Confidence scores: Detection confidence (0–1)
- Class labels: What was detected (e.g., "person", "sports_ball")

In [ ]:
print(f"Processing {len(frames_color)} frames with YOLO detection...")
start_time = time.time()

tracked_results = run_yolo_detection(model, frames_color)

end_time = time.time()
print(f"Detection complete")
print(f"Total detections: {sum(len(r.boxes) for r in tracked_results)}")
print(f"Processing time: {end_time - start_time:.2f} seconds")

## Extract Raw Detection Data

Parse all tracking results into a structured format containing bounding boxes, track IDs, confidence scores, and class labels for each frame.

In [ ]:
detection_output = yolo_to_detection_output(tracked_results, model, camera_id=CURRENT_CAMERA_ID, fps=25)

# Show the first frame results for visualization
print(f"First frame detections: {detection_output.frames[0].num_detections} objects detected")
print(f"First frame detection details:")
for i, detection in enumerate(detection_output.frames[0].detections[:3]):  # Show details for the first 5 detections
    print(f"  Detection {i+1}: Class={detection.class_name}, Confidence={detection.confidence:.2f}, BBox={detection.bbox}")
show_images(tracked_results, is_yolo=True)

In [ ]:
produce_detection_output_video(frames_color, detection_output, f"results/detection/{CURRENT_CAMERA_ID}/detections.mp4")